# p53 Mutant Stability Analysis - Explicit Solvent MD (Multi-GPU)

This notebook compares the stability of:
- **Wild-type (WT)** p53 core domain (baseline)
- **R175H** cancer mutation (destabilized)
- **R175H + N239Y** rescue mutation combination

## Method
- **Explicit solvent (TIP3P)** - More accurate than implicit solvent
- **2 fs timestep** - Standard for explicit solvent with HBonds constraints
- **NPT ensemble** - Constant pressure and temperature
- **ESMFold API** - Structure prediction without local installation
- **Multi-GPU parallel** - Runs variants simultaneously on available GPUs

In [ ]:
# @title 1. Install Dependencies
!pip install -q openmm requests numpy matplotlib mdtraj
!pip install -q --use-pep517 git+https://github.com/openmm/pdbfixer.git

In [ ]:
# @title 2. Imports and GPU Detection
import requests
import time
import os
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

from pdbfixer import PDBFixer
from openmm import *
from openmm.app import *
from openmm.unit import *
import mdtraj as md

print("All imports successful!")

# Detect available GPUs
def detect_gpus():
    """Detect available CUDA GPUs."""
    n_platforms = Platform.getNumPlatforms()
    platforms = [Platform.getPlatform(i).getName() for i in range(n_platforms)]
    print(f"Available OpenMM platforms: {platforms}")
    
    # Check nvidia-smi first (most reliable)
    try:
        result = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            gpu_lines = [l for l in result.stdout.strip().split('\n') if l.startswith('GPU')]
            n_gpus = len(gpu_lines)
            print(f"\nnvidia-smi detected {n_gpus} GPU(s):")
            for line in gpu_lines:
                print(f"  {line}")
            
            if 'CUDA' in platforms:
                return n_gpus, 'CUDA'
            elif 'OpenCL' in platforms:
                return n_gpus, 'OpenCL'
    except Exception as e:
        print(f"nvidia-smi check failed: {e}")
    
    # Fallback: Check CUDA platform
    if 'CUDA' in platforms:
        print("CUDA platform available, assuming 1 GPU")
        return 1, 'CUDA'
    elif 'OpenCL' in platforms:
        print("OpenCL platform available")
        return 1, 'OpenCL'
    else:
        print("No GPU found, using CPU")
        return 0, 'CPU'

N_GPUS, PLATFORM_NAME = detect_gpus()
print(f"\n>>> Will use: {PLATFORM_NAME} with {max(1, N_GPUS)} device(s) <<<")

# Verify GPU access
if PLATFORM_NAME == 'CUDA' and N_GPUS >= 1:
    print("\nVerifying GPU access...")
    try:
        platform = Platform.getPlatformByName('CUDA')
        for i in range(N_GPUS):
            props = {'DeviceIndex': str(i)}
            # Create a minimal test context
            system = System()
            system.addParticle(1.0)
            integrator = VerletIntegrator(0.001)
            context = Context(system, integrator, platform, props)
            del context
            print(f"  GPU {i}: OK")
        print("All GPUs verified!")
    except Exception as e:
        print(f"  GPU verification failed: {e}")
        print("  Falling back to single GPU mode")
        N_GPUS = 1

In [ ]:
# @title 3. Configuration

# === p53 Sequence ===
P53_FULL = (
    "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP"
    "DEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAK"
    "SVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHE"
    "RCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNS"
    "SCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELP"
    "PGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPG"
    "GSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD"
)
P53_CORE = P53_FULL[93:312]  # Core domain (residues 94-312)
CORE_START = 94

# === Simulation Parameters (Explicit Solvent) ===
TIMESTEP = 2.0                  # fs (standard for explicit solvent)
EQUILIBRATION_STEPS = 50000     # 100 ps NPT equilibration
PRODUCTION_STEPS = 500000       # 1 ns production (increase for better sampling)
SAVE_INTERVAL = 1000            # Save every 2 ps

# === Variants to simulate ===
VARIANTS = {
    'WT': [],                          # Wild-type (no mutations)
    'R175H': ['R175H'],                # Cancer mutation only
    'R175H_N239Y': ['R175H', 'N239Y']  # Cancer + rescue mutation
}

# === ESMFold API ===
ESMFOLD_API_URL = "https://api.esmatlas.com/foldSequence/v1/pdb/"

# Create output directories
os.makedirs("structures", exist_ok=True)
os.makedirs("trajectories", exist_ok=True)

print(f"p53 core domain: {len(P53_CORE)} residues")
print(f"Variants to simulate: {list(VARIANTS.keys())}")
print(f"\nSimulation parameters:")
print(f"  Solvent: Explicit (TIP3P)")
print(f"  Timestep: {TIMESTEP} fs")
print(f"  Equilibration: {EQUILIBRATION_STEPS * TIMESTEP / 1000:.0f} ps")
print(f"  Production: {PRODUCTION_STEPS * TIMESTEP / 1e6:.1f} ns per variant")

In [ ]:
# @title 4. Helper Functions

def apply_mutations(sequence, mutations):
    """Apply mutations to the p53 core domain sequence."""
    for mut in mutations:
        wt_aa = mut[0]
        pos = int(mut[1:-1])
        mut_aa = mut[-1]
        core_pos = pos - CORE_START

        if sequence[core_pos] != wt_aa:
            raise ValueError(f"Expected {wt_aa} at position {pos}, found {sequence[core_pos]}")

        sequence = sequence[:core_pos] + mut_aa + sequence[core_pos+1:]
        print(f"  Applied {mut} at core position {core_pos}")
    return sequence


def predict_structure_esmfold(sequence, max_retries=3):
    """Predict structure using ESMFold API with retry logic."""
    for attempt in range(max_retries):
        try:
            print(f"  Calling ESMFold API (attempt {attempt + 1}/{max_retries})...")
            response = requests.post(
                ESMFOLD_API_URL,
                data=sequence,
                headers={'Content-Type': 'text/plain'},
                timeout=300
            )
            if response.status_code == 200:
                print("  Success!")
                return response.text
            elif response.status_code == 503:
                print(f"  Server busy, waiting 30s...")
                time.sleep(30)
            else:
                print(f"  Error: {response.status_code}")
                time.sleep(10)
        except requests.Timeout:
            print(f"  Timeout, retrying...")
            time.sleep(10)
    raise RuntimeError("ESMFold API failed after all retries")


print("Helper functions defined.")

In [ ]:
# @title 5. Explicit Solvent MD Simulation Function (GPU-aware)

# Thread lock for printing
print_lock = threading.Lock()

def thread_print(*args, **kwargs):
    """Thread-safe print."""
    with print_lock:
        print(*args, **kwargs)

def run_simulation(name, mutations, gpu_id=0):
    """
    Run MD simulation with explicit solvent (TIP3P) on specified GPU.
    
    Args:
        name: Variant name (e.g., 'WT', 'R175H')
        mutations: List of mutations to apply
        gpu_id: GPU device index (0, 1, etc.)
    
    Returns:
        Dictionary with simulation results
    """
    thread_print(f"\n{'='*60}")
    thread_print(f"SIMULATING: {name} (Explicit Solvent) on GPU {gpu_id}")
    thread_print(f"{'='*60}")
    
    # 1. Apply mutations to sequence
    thread_print(f"\n[{name}] Preparing sequence...")
    sequence = P53_CORE
    if mutations:
        sequence = apply_mutations(sequence, mutations)
    else:
        thread_print("  No mutations (wild-type)")
    
    # 2. Get structure from ESMFold (use cache if available)
    pdb_path = f"structures/{name}_esmfold.pdb"
    if not os.path.exists(pdb_path):
        thread_print(f"\n[{name}] Predicting structure with ESMFold...")
        pdb_string = predict_structure_esmfold(sequence)
        with open(pdb_path, 'w') as f:
            f.write(pdb_string)
    else:
        thread_print(f"\n[{name}] Using cached structure: {pdb_path}")
    
    # 3. Fix structure with PDBFixer
    thread_print(f"\n[{name}] Fixing structure with PDBFixer...")
    fixer = PDBFixer(filename=pdb_path)
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)
    thread_print(f"  [{name}] Fixed structure: {fixer.topology.getNumAtoms()} atoms")
    
    # 4. Create explicit solvent system
    thread_print(f"\n[{name}] Creating explicit solvent system...")
    forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
    
    # Add solvent
    modeller = Modeller(fixer.topology, fixer.positions)
    thread_print(f"  [{name}] Adding water box (1.0 nm padding)...")
    modeller.addSolvent(
        forcefield,
        model='tip3p',
        padding=1.0*nanometer,
        ionicStrength=0.15*molar
    )
    thread_print(f"  [{name}] Solvated system: {modeller.topology.getNumAtoms()} atoms")
    
    # Save solvated structure
    solvated_path = f"structures/{name}_solvated.pdb"
    with open(solvated_path, 'w') as f:
        PDBFile.writeFile(modeller.topology, modeller.positions, f)
    
    # Create system
    thread_print(f"  [{name}] Creating OpenMM system...")
    system = forcefield.createSystem(
        modeller.topology,
        nonbondedMethod=PME,
        nonbondedCutoff=1.0*nanometer,
        constraints=HBonds
    )
    
    # Setup platform with specific GPU
    if PLATFORM_NAME == 'CUDA':
        platform = Platform.getPlatformByName('CUDA')
        properties = {'DeviceIndex': str(gpu_id), 'Precision': 'mixed'}
        thread_print(f"  [{name}] Using CUDA GPU {gpu_id}")
    elif PLATFORM_NAME == 'OpenCL':
        platform = Platform.getPlatformByName('OpenCL')
        properties = {'DeviceIndex': str(gpu_id), 'Precision': 'mixed'}
        thread_print(f"  [{name}] Using OpenCL device {gpu_id}")
    else:
        platform = Platform.getPlatformByName('CPU')
        properties = {}
        thread_print(f"  [{name}] Using CPU")
    
    # 5. Energy minimization
    thread_print(f"\n[{name}] Energy minimization...")
    integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, TIMESTEP*femtoseconds)
    
    if properties:
        simulation = Simulation(modeller.topology, system, integrator, platform, properties)
    else:
        simulation = Simulation(modeller.topology, system, integrator, platform)
    
    simulation.context.setPositions(modeller.positions)
    
    e_initial = simulation.context.getState(getEnergy=True).getPotentialEnergy()
    simulation.minimizeEnergy(maxIterations=1000)
    e_final = simulation.context.getState(getEnergy=True).getPotentialEnergy()
    thread_print(f"  [{name}] Energy: {e_initial} -> {e_final}")
    
    positions = simulation.context.getState(getPositions=True).getPositions()
    
    # 6. NPT Equilibration
    thread_print(f"\n[{name}] NPT Equilibration ({EQUILIBRATION_STEPS * TIMESTEP / 1000:.0f} ps)...")
    
    # Add barostat for pressure control
    system.addForce(MonteCarloBarostat(1*bar, 300*kelvin))
    
    integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, TIMESTEP*femtoseconds)
    if properties:
        simulation = Simulation(modeller.topology, system, integrator, platform, properties)
    else:
        simulation = Simulation(modeller.topology, system, integrator, platform)
    simulation.context.setPositions(positions)
    simulation.context.setVelocitiesToTemperature(300*kelvin)
    
    # Progress reporter for equilibration (reduced frequency for parallel runs)
    log_file = f"trajectories/{name}_eq_log.txt"
    simulation.reporters.append(
        StateDataReporter(log_file, 10000, step=True, temperature=True, 
                          progress=True, remainingTime=True, speed=True,
                          totalSteps=EQUILIBRATION_STEPS)
    )
    
    simulation.step(EQUILIBRATION_STEPS)
    
    # Save equilibrated structure
    eq_positions = simulation.context.getState(getPositions=True).getPositions()
    eq_path = f"structures/{name}_equilibrated.pdb"
    with open(eq_path, 'w') as f:
        PDBFile.writeFile(modeller.topology, eq_positions, f)
    thread_print(f"  [{name}] Saved: {eq_path}")
    
    # 7. Production MD
    thread_print(f"\n[{name}] Production MD ({PRODUCTION_STEPS * TIMESTEP / 1e6:.1f} ns) on GPU {gpu_id}...")
    
    # Clear reporters and add new ones
    simulation.reporters.clear()
    
    traj_path = f"trajectories/{name}_traj.dcd"
    log_path = f"trajectories/{name}_log.csv"
    
    # Remove old trajectory if exists
    if os.path.exists(traj_path):
        os.remove(traj_path)
    
    simulation.reporters.append(DCDReporter(traj_path, SAVE_INTERVAL))
    simulation.reporters.append(
        StateDataReporter(log_path, SAVE_INTERVAL, step=True, time=True,
                          potentialEnergy=True, temperature=True)
    )
    # Progress to file (less cluttered for parallel runs)
    simulation.reporters.append(
        StateDataReporter(f"trajectories/{name}_progress.txt", 50000, step=True, time=True,
                          potentialEnergy=True, temperature=True,
                          progress=True, remainingTime=True, speed=True,
                          totalSteps=PRODUCTION_STEPS)
    )
    
    simulation.step(PRODUCTION_STEPS)
    
    # 8. Analysis
    thread_print(f"\n[{name}] Analyzing trajectory...")
    traj = md.load(traj_path, top=eq_path)
    
    # Select protein atoms only (exclude water)
    protein_idx = traj.topology.select('protein')
    protein_traj = traj.atom_slice(protein_idx)
    thread_print(f"  [{name}] Protein atoms: {protein_traj.n_atoms}")
    
    # RMSD (protein only)
    rmsd = md.rmsd(protein_traj, protein_traj, 0) * 10  # nm to Angstrom
    
    # RMSF (CA atoms)
    protein_aligned = protein_traj.superpose(protein_traj, 0)
    ca_indices = protein_traj.topology.select('name CA')
    rmsf = md.rmsf(protein_aligned, protein_aligned, frame=0, atom_indices=ca_indices) * 10
    residue_nums = [protein_traj.topology.atom(i).residue.resSeq for i in ca_indices]
    
    # Save analysis
    np.save(f"trajectories/{name}_rmsd.npy", rmsd)
    np.save(f"trajectories/{name}_rmsf.npy", rmsf)
    
    results = {
        'name': name,
        'mutations': mutations,
        'gpu_id': gpu_id,
        'n_frames': traj.n_frames,
        'n_atoms': protein_traj.n_atoms,
        'n_atoms_total': traj.n_atoms,
        'rmsd': rmsd,
        'rmsf': rmsf,
        'residue_nums': residue_nums,
        'mean_rmsd': np.mean(rmsd),
        'final_rmsd': rmsd[-1],
        'max_rmsd': np.max(rmsd),
        'mean_rmsf': np.mean(rmsf)
    }
    
    thread_print(f"\n[{name}] Complete on GPU {gpu_id}!")
    thread_print(f"  [{name}] Frames: {traj.n_frames}")
    thread_print(f"  [{name}] Mean RMSD: {np.mean(rmsd):.2f} A")
    thread_print(f"  [{name}] Final RMSD: {rmsd[-1]:.2f} A")
    
    return results


print("Simulation function defined (explicit solvent, multi-GPU).")

In [ ]:
# @title 6. Run All Simulations (Multi-GPU Parallel)

print("="*60)
print("RUNNING EXPLICIT SOLVENT MD SIMULATIONS (MULTI-GPU)")
print("="*60)
print(f"Method: Explicit solvent (TIP3P) + PME")
print(f"Timestep: {TIMESTEP} fs")
print(f"Equilibration: {EQUILIBRATION_STEPS * TIMESTEP / 1000:.0f} ps NPT")
print(f"Production: {PRODUCTION_STEPS * TIMESTEP / 1e6:.1f} ns per variant")
print(f"Variants: {list(VARIANTS.keys())}")
print(f"GPUs available: {N_GPUS}")
print("="*60)

# Prepare variant list with GPU assignments
variant_list = list(VARIANTS.items())
n_variants = len(variant_list)

if N_GPUS >= 2:
    # Multi-GPU: Run variants in parallel
    print(f"\n>>> PARALLEL MODE: Using {N_GPUS} GPUs simultaneously <<<")
    print(f"    GPU 0: WT, R175H_N239Y (if 3 variants)")
    print(f"    GPU 1: R175H")
    print("="*60)
    
    results = {}
    
    # First batch: Run 2 variants in parallel (one per GPU)
    with ThreadPoolExecutor(max_workers=N_GPUS) as executor:
        futures = {}
        
        # Assign variants to GPUs (round-robin)
        for i, (name, mutations) in enumerate(variant_list):
            gpu_id = i % N_GPUS
            future = executor.submit(run_simulation, name, mutations, gpu_id)
            futures[future] = name
            print(f"  Submitted {name} to GPU {gpu_id}")
        
        # Collect results as they complete
        for future in as_completed(futures):
            name = futures[future]
            try:
                results[name] = future.result()
                print(f"\n>>> {name} COMPLETED <<<")
            except Exception as e:
                print(f"\n>>> {name} FAILED: {e} <<<")
                raise

else:
    # Single GPU or CPU: Run sequentially
    print(f"\n>>> SEQUENTIAL MODE: Using single device <<<")
    print("="*60)
    
    results = {}
    for name, mutations in VARIANTS.items():
        results[name] = run_simulation(name, mutations, gpu_id=0)

print("\n" + "="*60)
print("ALL SIMULATIONS COMPLETE")
print("="*60)

# Print timing summary
for name, r in results.items():
    print(f"  {name}: GPU {r.get('gpu_id', 0)}, Final RMSD: {r['final_rmsd']:.2f} A")

In [ ]:
# @title 7. Generate Comparison Plots

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {'WT': 'green', 'R175H': 'red', 'R175H_N239Y': 'blue'}
labels = {
    'WT': 'Wild-type (baseline)',
    'R175H': 'R175H (cancer mutation)',
    'R175H_N239Y': 'R175H+N239Y (rescue)'
}

# Plot 1: RMSD over time
ax1 = axes[0, 0]
for name, r in results.items():
    time_ns = np.arange(len(r['rmsd'])) * TIMESTEP * SAVE_INTERVAL / 1e6
    ax1.plot(time_ns, r['rmsd'], color=colors[name], linewidth=1.5, label=labels[name])

ax1.axhline(y=2.5, color='orange', linestyle='--', linewidth=2, label='Stability threshold (2.5 A)')
ax1.set_xlabel('Time (ns)', fontsize=12)
ax1.set_ylabel('RMSD (A)', fontsize=12)
ax1.set_title('RMSD Comparison Over Time (Explicit Solvent)', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Plot 2: Final RMSD bar chart
ax2 = axes[0, 1]
names = list(results.keys())
final_rmsds = [results[n]['final_rmsd'] for n in names]
bar_colors = [colors[n] for n in names]

bars = ax2.bar(names, final_rmsds, color=bar_colors, alpha=0.7, edgecolor='black', linewidth=2)
ax2.axhline(y=2.5, color='orange', linestyle='--', linewidth=2, label='Stability threshold')

for bar, val in zip(bars, final_rmsds):
    ax2.text(bar.get_x() + bar.get_width()/2., val + 0.15,
             f'{val:.2f} A', ha='center', fontsize=11, fontweight='bold')

ax2.set_ylabel('Final RMSD (A)', fontsize=12)
ax2.set_title('Final RMSD Comparison', fontsize=14, fontweight='bold')
ax2.legend()
ax2.set_ylim(0, max(final_rmsds) * 1.4)

# Plot 3: RMSF comparison
ax3 = axes[1, 0]
for name, r in results.items():
    ax3.plot(r['residue_nums'], r['rmsf'], color=colors[name],
             linewidth=1, label=labels[name], alpha=0.8)

# Mark mutation sites
ax3.axvline(x=175, color='red', linestyle=':', linewidth=2, alpha=0.7, label='R175H site')
ax3.axvline(x=239, color='blue', linestyle=':', linewidth=2, alpha=0.7, label='N239Y site')

ax3.set_xlabel('Residue Number', fontsize=12)
ax3.set_ylabel('RMSF (A)', fontsize=12)
ax3.set_title('Per-Residue Flexibility (RMSF)', fontsize=14, fontweight='bold')
ax3.legend(loc='upper right')
ax3.grid(True, alpha=0.3)

# Plot 4: Summary table
ax4 = axes[1, 1]
ax4.axis('off')

table_data = [['Variant', 'Mean RMSD', 'Final RMSD', 'Max RMSD', 'Status']]
for name in names:
    r = results[name]
    status = 'STABLE' if r['final_rmsd'] < 2.5 else 'UNSTABLE'
    table_data.append([
        labels[name],
        f"{r['mean_rmsd']:.2f} A",
        f"{r['final_rmsd']:.2f} A",
        f"{r['max_rmsd']:.2f} A",
        status
    ])

table = ax4.table(cellText=table_data, loc='center', cellLoc='center',
                  colWidths=[0.35, 0.15, 0.15, 0.15, 0.15])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2)

# Style header row
for j in range(5):
    table[(0, j)].set_facecolor('#4472C4')
    table[(0, j)].set_text_props(color='white', fontweight='bold')

# Color status cells
for i, name in enumerate(names, 1):
    if results[name]['final_rmsd'] < 2.5:
        table[(i, 4)].set_facecolor('#C6EFCE')
    else:
        table[(i, 4)].set_facecolor('#FFC7CE')

ax4.set_title('Simulation Summary (Explicit Solvent)', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('trajectories/comparison_analysis_explicit.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved to: trajectories/comparison_analysis_explicit.png")

In [ ]:
# @title 8. Final Results Summary

print("="*60)
print("FINAL RESULTS SUMMARY (EXPLICIT SOLVENT)")
print("="*60)

print(f"\nSimulation Parameters:")
print(f"  Method: Explicit solvent (TIP3P + PME)")
print(f"  Timestep: {TIMESTEP} fs")
print(f"  Equilibration: {EQUILIBRATION_STEPS * TIMESTEP / 1000:.0f} ps NPT")
print(f"  Production: {PRODUCTION_STEPS * TIMESTEP / 1e6:.1f} ns")
print(f"  Frames per simulation: {results['WT']['n_frames']}")
print(f"  Total atoms (with water): ~{results['WT']['n_atoms_total']}")

print(f"\nStability Assessment (threshold: 2.5 A):")
print("-" * 60)

for name in ['WT', 'R175H', 'R175H_N239Y']:
    r = results[name]
    status = "STABLE" if r['final_rmsd'] < 2.5 else "UNSTABLE"
    status_icon = "[OK]" if r['final_rmsd'] < 2.5 else "[!!]"

    print(f"\n{labels[name]}:")
    print(f"  Final RMSD:  {r['final_rmsd']:.2f} A")
    print(f"  Mean RMSD:   {r['mean_rmsd']:.2f} A")
    print(f"  Max RMSD:    {r['max_rmsd']:.2f} A")
    print(f"  Status:      {status} {status_icon}")

print("\n" + "="*60)
print("INTERPRETATION")
print("="*60)

wt_rmsd = results['WT']['final_rmsd']
r175h_rmsd = results['R175H']['final_rmsd']
rescue_rmsd = results['R175H_N239Y']['final_rmsd']

if r175h_rmsd > wt_rmsd:
    print(f"\n1. R175H mutation DESTABILIZES p53")
    print(f"   WT: {wt_rmsd:.2f} A -> R175H: {r175h_rmsd:.2f} A")
    print(f"   Destabilization: +{r175h_rmsd - wt_rmsd:.2f} A")
else:
    print(f"\n1. R175H mutation effect unclear (may need longer simulation)")

if rescue_rmsd < r175h_rmsd:
    improvement = ((r175h_rmsd - rescue_rmsd) / r175h_rmsd) * 100
    print(f"\n2. N239Y RESCUES stability ({improvement:.1f}% improvement)")
    print(f"   R175H: {r175h_rmsd:.2f} A -> R175H+N239Y: {rescue_rmsd:.2f} A")
    if rescue_rmsd < 2.5:
        print(f"   Structure is now STABLE!")
else:
    print(f"\n2. N239Y rescue effect not observed")
    print(f"   Consider: longer simulation, different rescue mutations")

print("\n" + "="*60)
print("OUTPUT FILES")
print("="*60)
print("\nStructures:")
for name in names:
    print(f"  structures/{name}_solvated.pdb")
    print(f"  structures/{name}_equilibrated.pdb")
print("\nTrajectories:")
for name in names:
    print(f"  trajectories/{name}_traj.dcd")
    print(f"  trajectories/{name}_log.csv")
print("\nAnalysis:")
print(f"  trajectories/comparison_analysis_explicit.png")

## Notes on Explicit Solvent Simulations

### Advantages over Implicit Solvent
- **More accurate** - Explicit water molecules capture hydrogen bonding and hydration effects
- **Better for stability** - More reliable RMSD measurements
- **Industry standard** - Used in all publication-quality MD studies

### Computational Cost
- System size: ~100,000 atoms (vs ~3,500 for implicit)
- Speed: ~30-50x slower than implicit solvent
- Recommended: Use GPU acceleration (Colab T4 or better)

### For Best Results
1. Run 10-50 ns production (increase PRODUCTION_STEPS)
2. Run 3-5 independent replicates (different random seeds)
3. Analyze last 50% of trajectory (after equilibration)
4. Compare with experimental melting temperatures if available

### References
- AMBER14 forcefield: Maier et al., JCTC 2015
- TIP3P water model: Jorgensen et al., JCP 1983
- p53 stability: Bullock et al., PNAS 2000